# Full Atlas-Free ALE3DCNN Pipeline

This notebook is the single playbook for the atlas-free CNN workflow.

Pipeline:

```text
Stage 0: pack PubMed + Nilearn + NeuroVault into HF-friendly CNN tensors/text pairs
Stage 1: train atlas-free CNN autoencoder
Stage 1 eval: reconstruction capability on train/val/test
Stage 2: initialize encoder from the autoencoder encoder
Stage 3: contrastive fine-tuning for brain-to-text retrieval
Stage 3 eval: brain-to-text and text-to-brain retrieval metrics
Stage 4: text-to-brain generation evaluation on held-out test set
Optional Stage 4b: train a text-to-brain projection head on train only, tune on val, evaluate on test
```

**Hard rule:** held-out test-set text-to-brain generation is evaluation, not training. Any additional text-to-brain projection head is trained on train, selected/tuned on val, and evaluated once on test.

## Colab Setup

Run this cell first in Colab. It clones the repo, installs the package, and switches the working directory to the repo root. The next setup cell mounts Google Drive and saves model checkpoints, plots, evaluation files, and summaries under `MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline`. If you are already running inside a local checkout, it leaves your checkout alone.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_REF = 'neurovlm_gnn'
COLAB_REPO = Path('/content/neurovlm')
IN_COLAB = Path('/content').exists() and 'google.colab' in sys.modules

def _has_repo_root(path):
    return (Path(path) / 'pyproject.toml').exists() and (Path(path) / 'src/neurovlm').exists()

if IN_COLAB:
    if not _has_repo_root(COLAB_REPO):
        subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(COLAB_REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(COLAB_REPO), 'fetch', 'origin', REPO_REF], check=False)
        subprocess.run(['git', '-C', str(COLAB_REPO), 'checkout', REPO_REF], check=False)
        subprocess.run(['git', '-C', str(COLAB_REPO), 'pull', '--ff-only', 'origin', REPO_REF], check=False)
    os.chdir(COLAB_REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
else:
    print('Not in Colab, using the current local environment.')

# `pip install -e .` installs src/neurovlm, while this experiment package lives at repo root.
REPO_ROOT = Path.cwd()
for import_root in [REPO_ROOT, REPO_ROOT / 'src']:
    import_root = str(import_root)
    if import_root not in sys.path:
        sys.path.insert(0, import_root)

git_head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], text=True, capture_output=True)
print('cwd:', Path.cwd())
print('repo ref:', REPO_REF)
print('git head:', git_head.stdout.strip() or 'unknown')
print('repo-root experiment package exists:', (REPO_ROOT / 'atlas_free_multipositive/training').exists())


In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import torch

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    return start

ROOT = find_repo_root()
for import_root in [ROOT, ROOT / 'src']:
    import_root = str(import_root)
    if import_root not in sys.path:
        sys.path.insert(0, import_root)

CACHE = ROOT / 'atlas_free_multipositive/cache'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
IN_COLAB = Path('/content').exists() and 'google.colab' in sys.modules

drive_mounted = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_mounted = Path('/content/drive/MyDrive').exists()
    except Exception as exc:
        print('Google Drive mount skipped:', repr(exc))

if drive_mounted:
    DRIVE_ROOT = Path('/content/drive/MyDrive/neurovlm')
    RUNS_DIR = DRIVE_ROOT / 'runs_atlas_free_cnn_full_pipeline'
    EVAL_RESOURCE_DIR = Path('/content/drive/MyDrive/neurovlm_evaluation_resources')
    OUT = RUNS_DIR / f'full_atlas_free_cnn_{RUN_STAMP}'
else:
    DRIVE_ROOT = ROOT / 'experiments/3dcnn/outputs'
    RUNS_DIR = DRIVE_ROOT / 'runs_atlas_free_cnn_full_pipeline'
    EVAL_RESOURCE_DIR = ROOT / 'experiments/evaluation_resources'
    OUT = RUNS_DIR / f'full_atlas_free_cnn_{RUN_STAMP}'

for path in [DRIVE_ROOT, RUNS_DIR, EVAL_RESOURCE_DIR, OUT]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGET_SHAPE = (36, 45, 38)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TEST_IS_EVALUATION_ONLY = True
run_config = {
    'device': str(DEVICE),
    'target_shape': TARGET_SHAPE,
    'test_is_eval_only': TEST_IS_EVALUATION_ONLY,
    'root': str(ROOT),
    'drive_root': str(DRIVE_ROOT),
    'runs_dir': str(RUNS_DIR),
    'eval_resource_dir': str(EVAL_RESOURCE_DIR),
    'out': str(OUT),
    'run_stamp': RUN_STAMP,
}
(OUT / 'run_config.json').write_text(json.dumps(run_config, indent=2))
print(run_config)

## Stage 0: Pack Data For Local/Hugging Face Use

Run this after building PubMed/Nilearn split JSONL files and the staged NeuroVault collector. NeuroVault defaults to deterministic train/val/test splitting, so it does not all leak into train.

The packed output is intended for Hugging Face upload:

```text
atlas_free_cnn_volumes.pt
atlas_free_cnn_rows.parquet
atlas_free_cnn_text_pairs.parquet
atlas_free_cnn_manifest.json
```

The text-pair parquet includes `is_primary_text`. Primary text is selected by metadata specificity, not by model similarity. For this single-text 3DCNN experiment the intended primary policy is: PubMed title + summary/abstract-style text; NeuroVault image title + description or collection title + description, falling back to task/contrast label; Nilearn atlas/network label.

In [ ]:
# Local packing command. Run in a terminal/local checkout, not in Colab.
# This creates the files that should be uploaded to Hugging Face.
# !.conda/bin/python -m atlas_free_multipositive.data_building.pack_atlas_free_cnn_dataset \
#   --jsonl atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl \
#           atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl \
#           atlas_free_multipositive/cache/unified_jsonl/splits/test.jsonl \
#   --neurovault-dir atlas_free_multipositive/cache/neurovault_diverse_5k \
#   --neurovault-split auto \
#   --neurovault-max-per-collection 50 \
#   --strong-neurovault-only \
#   --output-dir atlas_free_multipositive/cache/hf_atlas_free_cnn

PACKED_DIR = CACHE / 'hf_atlas_free_cnn'
LOCAL_VOLUME_PT = PACKED_DIR / 'atlas_free_cnn_volumes.pt'
LOCAL_ROWS = PACKED_DIR / 'atlas_free_cnn_rows.parquet'
LOCAL_TEXT_PAIRS = PACKED_DIR / 'atlas_free_cnn_text_pairs.parquet'
LOCAL_MANIFEST = PACKED_DIR / 'atlas_free_cnn_manifest.json'

if LOCAL_MANIFEST.exists():
    print(json.loads(LOCAL_MANIFEST.read_text()))
else:
    print('Local packed dataset not found. In Colab this is expected; use Hugging Face loading below.')
    print('Missing:', PACKED_DIR)


## Colab/Hugging Face Loading

Colab should load the packed dataset from Hugging Face. Local files under `/content/atlas_free_multipositive/cache/...` are not expected to exist unless you manually copied them there.

Expected HF dataset repo:

```text
neurovlm/atlas_free_cnn_dataset
```


In [ ]:
HF_DATASET_REPO = 'neurovlm/atlas_free_cnn_dataset'
USE_HF_DATA = True  # Colab default. Set False only in a local checkout with packed files present.

def _local_packed_files_exist():
    return all(p.exists() for p in [LOCAL_VOLUME_PT, LOCAL_ROWS, LOCAL_TEXT_PAIRS, LOCAL_MANIFEST])

def load_packed_dataset(prefer_hf=True, repo_id=HF_DATASET_REPO):
    if prefer_hf:
        try:
            from huggingface_hub import hf_hub_download
            volume_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_volumes.pt', repo_type='dataset')
            rows_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_rows.parquet', repo_type='dataset')
            pairs_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_text_pairs.parquet', repo_type='dataset')
            manifest_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_manifest.json', repo_type='dataset')
            print('Loaded packed atlas-free CNN dataset from Hugging Face:', repo_id)
            payload = torch.load(volume_path, map_location='cpu', weights_only=False)
            return {
                **payload,
                'rows': pd.read_parquet(rows_path),
                'text_pairs': pd.read_parquet(pairs_path),
                'manifest': json.loads(Path(manifest_path).read_text()),
            }
        except Exception as exc:
            print('HF load failed:', repr(exc))
            print('If this is a private repo, run `huggingface-cli login` or set an HF token.')

    if not _local_packed_files_exist():
        missing = [str(p) for p in [LOCAL_VOLUME_PT, LOCAL_ROWS, LOCAL_TEXT_PAIRS, LOCAL_MANIFEST] if not p.exists()]
        raise FileNotFoundError('Packed dataset not found locally. Use USE_HF_DATA=True in Colab, or upload/copy packed files. Missing: ' + '; '.join(missing))

    print('Loaded packed atlas-free CNN dataset from local files:', PACKED_DIR)
    payload = torch.load(LOCAL_VOLUME_PT, map_location='cpu', weights_only=False)
    return {
        **payload,
        'rows': pd.read_parquet(LOCAL_ROWS),
        'text_pairs': pd.read_parquet(LOCAL_TEXT_PAIRS),
        'manifest': json.loads(LOCAL_MANIFEST.read_text()),
    }

packed = load_packed_dataset(prefer_hf=USE_HF_DATA)
volumes = packed['volumes'].float()
rows = packed['rows'].copy()
text_pairs = packed['text_pairs'].copy()
print(volumes.shape, rows.shape, text_pairs.shape)
print(rows['split'].value_counts())
print(rows['source'].value_counts().head(20))


## Materialize JSONL Splits For Existing Trainers

The current training scripts expect JSONL rows with `tensor_path`, `tensor_index`, and `positive_texts`. For this 3DCNN experiment we are **not** doing multi-positive text yet, so each map gets exactly one primary text target selected by metadata priority: PubMed title + summary/abstract-style text; NeuroVault title + description if available, otherwise task label; Nilearn atlas/network label. The test split is written for evaluation loaders only.

In [ ]:
def materialize_primary_jsonl_splits_from_packed(packed, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    tensor_path = out_dir / 'atlas_free_cnn_volumes.pt'
    torch.save({'volumes': packed['volumes'], 'map_ids': packed.get('map_ids')}, tensor_path)
    rows_df = packed['rows'].copy()
    pairs_df = packed['text_pairs'].copy()
    pairs_df = pairs_df[pairs_df['is_primary_text']].copy()
    written = {}
    for split, split_rows in rows_df.groupby('split'):
        path = out_dir / f'{split}.jsonl'
        with path.open('w') as f:
            for _, row in split_rows.iterrows():
                positives = pairs_df[pairs_df.map_id == row.map_id].sort_values('pair_rank').head(1)
                record = row.to_dict()
                record.update({
                    'tensor_path': str(tensor_path),
                    'tensor_index': int(row.tensor_index),
                    'positive_texts': positives[['text','term','category','source','weight','reliability']].to_dict('records'),
                    'primary_text_only': True,
                })
                f.write(json.dumps(record, default=str) + '\n')
        written[split] = path
    return written

SPLIT_JSONL = materialize_primary_jsonl_splits_from_packed(packed, CACHE / 'hf_atlas_free_cnn/materialized_primary_jsonl')
SPLIT_JSONL

## Stage 1: Train Atlas-Free CNN Autoencoder

Training objective:

```text
ALE volume -> CNN encoder -> latent -> CNN decoder -> reconstructed ALE volume
```

Train on train, select on val. Test is untouched until evaluation.

In [ ]:
training_file = ROOT / 'atlas_free_multipositive/training/train_autoencoder_sparse.py'
if not training_file.exists():
    raise FileNotFoundError(
        f'Missing {training_file}. The Colab checkout is probably older than this notebook. '
        'Pull/clone the current repo version, or upload the current atlas_free_multipositive folder.'
    )

try:
    from atlas_free_multipositive.training.train_autoencoder_sparse import train_from_config as train_autoencoder
except ModuleNotFoundError as exc:
    print('cwd:', Path.cwd())
    print('ROOT:', ROOT)
    print('first sys.path entries:', sys.path[:5])
    print('training file exists:', training_file.exists())
    raise exc

RUN_STAGE1_AUTOENCODER = True
RETRAIN_STAGE1_IF_PRESENT = False

ae_cfg = {
    'train_jsonl': str(SPLIT_JSONL['train']),
    'val_jsonl': str(SPLIT_JSONL['val']),
    'target_shape': list(TARGET_SHAPE),
    'output_dir': str(OUT / 'stage1_autoencoder'),
    'checkpoint_dir': str(OUT / 'stage1_autoencoder/checkpoints'),
    'batch_size': 8,
    'epochs': 20,
    'lr': 1e-4,
    'include_voxel_auroc': True,
    'progress': True,
    'model': {'latent_dim': 384, 'base_channels': 8, 'num_blocks': 2, 'encoder_arch': 'plain'},
    'loss': {'lambda_recon': 1.0, 'lambda_dice': 0.5, 'lambda_topk': 0.5, 'lambda_corr': 0.25},
    'weighted_recon': {'type': 'mse', 'alpha': 10.0, 'gamma': 1.0},
}

AE_CKPT = OUT / 'stage1_autoencoder/checkpoints/best_generation_top5_dice.pt'
if RUN_STAGE1_AUTOENCODER and (RETRAIN_STAGE1_IF_PRESENT or not AE_CKPT.exists()):
    print('Training Stage 1 autoencoder. Epoch logs and batch progress should appear below.')
    ae_result = train_autoencoder(ae_cfg)
    AE_CKPT = Path(ae_result['checkpoint_dir']) / 'best_generation_top5_dice.pt'
elif AE_CKPT.exists():
    print('Using existing Stage 1 checkpoint:', AE_CKPT)
else:
    raise FileNotFoundError('RUN_STAGE1_AUTOENCODER is False and no Stage 1 checkpoint exists: ' + str(AE_CKPT))

print('Stage 1 checkpoint:', AE_CKPT)
AE_CKPT

## Stage 1 Eval: Reconstruction Metrics And Random Qualitative Examples

Report MSE, MAE, weighted MSE/BCE, voxel AUROC, spatial Pearson correlation, top-1/5/10% Dice or overlap, and peak recall when coordinates are available. Qualitative examples must be random from val/test, not cherry-picked.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from torch.utils.data import DataLoader

from atlas_free_multipositive.evaluation.generation_metrics import generation_metrics, voxel_auroc
from atlas_free_multipositive.training.datasets import UnifiedMapTextDataset
from atlas_free_multipositive.training.train_autoencoder_sparse import VolumeCollator
from atlas_free_multipositive.training.model_wrappers import build_cnn_autoencoder, load_autoencoder_checkpoint

def mae(pred, target):
    return float((pred - target).abs().mean().item())

def evaluate_reconstruction_split(model, jsonl_path, split, batch_size=8):
    ds = UnifiedMapTextDataset(jsonl_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=VolumeCollator(TARGET_SHAPE))
    rows_out = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            x = batch['volume'].to(DEVICE)
            recon = torch.sigmoid(model(x))
            for i, map_id in enumerate(batch['map_id']):
                m = generation_metrics(recon[i:i+1].cpu(), x[i:i+1].cpu(), include_voxel_auroc=True)
                m['mae'] = mae(recon[i:i+1].cpu(), x[i:i+1].cpu())
                m.update({'map_id': map_id, 'split': split})
                rows_out.append(m)
    return pd.DataFrame(rows_out)

def _middle_slices(vol):
    arr = vol.squeeze().detach().cpu().numpy()
    return [arr[arr.shape[0]//2,:,:], arr[:,arr.shape[1]//2,:], arr[:,:,arr.shape[2]//2]]

def plot_random_reconstruction_examples(model, jsonl_path, split, out_png, out_dir, n=6):
    ds = UnifiedMapTextDataset(jsonl_path)
    idxs = random.sample(range(len(ds)), k=min(n, len(ds)))
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(len(idxs), 9, figsize=(18, 2.2 * len(idxs)))
    if len(idxs) == 1: axes = np.expand_dims(axes, 0)
    model.eval()
    for r, idx in enumerate(idxs):
        item = ds[idx]
        x = item['volume'].unsqueeze(0).to(DEVICE)
        with torch.no_grad(): recon = torch.sigmoid(model(x)).cpu()[0]
        true = item['volume'].cpu(); diff = (recon - true).abs()
        vmax = max(float(true.max()), float(recon.max()), 1e-6)
        metric = generation_metrics(recon.unsqueeze(0), true.unsqueeze(0), include_voxel_auroc=False)
        panels = _middle_slices(true) + _middle_slices(recon) + _middle_slices(diff)
        titles = ['orig axial','orig coronal','orig sagittal','recon axial','recon coronal','recon sagittal','diff axial','diff coronal','diff sagittal']
        for c, panel in enumerate(panels):
            axes[r, c].imshow(panel.T, origin='lower', cmap='magma', vmin=0, vmax=vmax if c < 6 else None)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            axes[r, c].set_title(titles[c], fontsize=8)
        axes[r, 0].set_ylabel(f"{split}\n{item['map_id']}\nr={metric['spatial_corr']:.2f} d5={metric['top5_dice']:.2f}", fontsize=8)
        fig.savefig(out_dir / f"{split}_{item['map_id']}.png", dpi=160, bbox_inches='tight')
    fig.tight_layout(); fig.savefig(out_png, dpi=180, bbox_inches='tight')
    return str(out_png), str(out_dir)

RUN_STAGE1_RECON_EVAL = True

recon_results = {}
reconstruction_plot_paths = {}
if RUN_STAGE1_RECON_EVAL:
    if not Path(AE_CKPT).exists():
        raise FileNotFoundError('Stage 1 checkpoint is missing, so reconstruction eval cannot run: ' + str(AE_CKPT))
    ae = build_cnn_autoencoder(
        TARGET_SHAPE,
        latent_dim=ae_cfg['model']['latent_dim'],
        base_channels=ae_cfg['model']['base_channels'],
        num_blocks=ae_cfg['model']['num_blocks'],
        encoder_arch=ae_cfg['model']['encoder_arch'],
    ).to(DEVICE)
    load_autoencoder_checkpoint(ae, AE_CKPT)
    for split in ['train', 'val', 'test']:
        print('Evaluating reconstruction split:', split)
        df = evaluate_reconstruction_split(ae, SPLIT_JSONL[split], split, batch_size=ae_cfg['batch_size'])
        df.to_csv(OUT / f'stage1_reconstruction_metrics_{split}.csv', index=False)
        recon_results[split] = df
    reconstruction_plot_paths['val'] = plot_random_reconstruction_examples(
        ae, SPLIT_JSONL['val'], 'val', OUT/'random_reconstruction_examples_val.png', OUT/'random_reconstruction_examples_val'
    )
    reconstruction_plot_paths['test'] = plot_random_reconstruction_examples(
        ae, SPLIT_JSONL['test'], 'test', OUT/'random_reconstruction_examples_test.png', OUT/'random_reconstruction_examples_test'
    )
    recon_summary = pd.DataFrame([
        {'split': split, **df.select_dtypes(include='number').mean().to_dict()}
        for split, df in recon_results.items()
    ])
    recon_summary.to_csv(OUT / 'stage1_reconstruction_summary.csv', index=False)
    print('Saved reconstruction outputs to:', OUT)
    display(recon_summary)
else:
    raise RuntimeError('RUN_STAGE1_RECON_EVAL is False. This notebook is configured to avoid silent no-op runs.')

## Stage 2-3: Encoder Init And Contrastive Fine-Tuning

Initialize the brain encoder from the autoencoder encoder, then fine-tune for single-primary-text contrastive retrieval. Evaluation must report both directions separately:

```text
brain -> text
text -> brain
```

Main metrics: exact PMID paper recall AUC, exact PMID normalized-k AUC, MeSH normalized-k AUC, semantic paper-style AUC, macro retrieval normalized-k AUC, network accuracy, network macro AUC, and network term normalized-k AUC.

In [ ]:
RUN_STAGE3_CONTRASTIVE = False
TEXT_EMBEDDING_CACHE = CACHE / 'text_embeddings/specter_text_cache.pt'
CONTRASTIVE_CKPT = OUT / 'stage3_contrastive/checkpoints/last.pt'

# Contrastive training entry point. Turn RUN_STAGE3_CONTRASTIVE on after creating/uploading TEXT_EMBEDDING_CACHE.
# !.conda/bin/python -m atlas_free_multipositive.training.train_contrastive \
#   --jsonl {SPLIT_JSONL['train']} \
#   --text-embeddings atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt \
#   --autoencoder-checkpoint {AE_CKPT} \
#   --batch-size 8 --steps 10000 --device cuda

def normalized_auc(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    if len(x) < 2: return float('nan')
    return float(np.trapz(y, x) / max(x[-1] - x[0], 1e-8))

def recall_curve(scores, positive_mask, k_values):
    order = torch.argsort(scores, dim=1, descending=True)
    curves = []
    for k in k_values:
        hit = positive_mask.gather(1, order[:, :k]).any(dim=1).float().mean().item()
        curves.append(hit)
    return curves

def bidirectional_retrieval_report(brain_emb, text_emb, positive_mask, k_values=None):
    brain_emb = torch.nn.functional.normalize(brain_emb.float(), dim=1)
    text_emb = torch.nn.functional.normalize(text_emb.float(), dim=1)
    scores_bt = brain_emb @ text_emb.T
    scores_tb = scores_bt.T
    if k_values is None:
        k_values = [1, 5, 10, 25, 50, 100, min(250, scores_bt.shape[1])]
    normalized_k = [k / scores_bt.shape[1] for k in k_values]
    bt = recall_curve(scores_bt, positive_mask.bool(), k_values)
    tb = recall_curve(scores_tb, positive_mask.T.bool(), k_values)
    mean_curve = [(a + b) / 2 for a, b in zip(bt, tb)]
    return {
        'k_values': k_values,
        'normalized_k_values': normalized_k,
        'brain_to_text_recall_curve': bt,
        'text_to_brain_recall_curve': tb,
        'mean_recall_curve': mean_curve,
        'brain_to_text_normalized_k_recall_curve_auc': normalized_auc(normalized_k, bt),
        'text_to_brain_normalized_k_recall_curve_auc': normalized_auc(normalized_k, tb),
        'normalized_k_recall_curve_auc': normalized_auc(normalized_k, mean_curve),
    }

if RUN_STAGE3_CONTRASTIVE:
    if not TEXT_EMBEDDING_CACHE.exists():
        raise FileNotFoundError('Stage 3 needs cached SPECTER/SPECTER2 embeddings: ' + str(TEXT_EMBEDDING_CACHE))
    cmd = [
        sys.executable, '-m', 'atlas_free_multipositive.training.train_contrastive',
        '--jsonl', str(SPLIT_JSONL['train']),
        '--text-embeddings', str(TEXT_EMBEDDING_CACHE),
        '--autoencoder-checkpoint', str(AE_CKPT),
        '--batch-size', '8', '--steps', '10000', '--device', str(DEVICE),
    ]
    print('Running Stage 3 contrastive training:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Stage 3 contrastive training is not run in this notebook yet. Set RUN_STAGE3_CONTRASTIVE=True after preparing TEXT_EMBEDDING_CACHE.')

# retrieval_report = bidirectional_retrieval_report(test_brain_emb, test_text_emb, test_positive_mask)
# json.dump(retrieval_report, open(OUT/'stage3_bidirectional_retrieval_curves.json','w'), indent=2)

## Stage 4: Held-Out Text-To-Brain Generation Evaluation

Default evaluation path:

```text
held-out test text -> SPECTER/SPECTER2 -> trained text projection -> CNN decoder -> generated ALE volume
```

This is evaluation on test. Do not fit anything in this stage.

In [ ]:
from atlas_free_multipositive.training.generation_baselines import global_mean_map, random_training_maps, nearest_neighbor_text_maps, category_average_maps, predict_category_average

def primary_text_table(rows, text_pairs, split):
    split_maps = rows[rows['split'] == split][['map_id','tensor_index','primary_text']]
    primary = text_pairs[(text_pairs['split'] == split) & (text_pairs['is_primary_text'])].copy()
    return split_maps.merge(primary[['map_id','text','term','category','source']], on='map_id', how='left')

def evaluate_generation_tensors(pred, target, batch_size=32):
    rows_out = []
    for start in range(0, len(target), batch_size):
        p = pred[start:start + batch_size].float()
        t = target[start:start + batch_size].float()
        m = generation_metrics(p, t, include_voxel_auroc=True)
        m['mae'] = mae(p, t)
        rows_out.append(m)
    return {k: float(np.nanmean([r[k] for r in rows_out])) for k in rows_out[0]}

def plot_random_generation_examples(true, generated, meta, out_png, out_dir, n=6):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    idxs = random.sample(range(len(true)), k=min(n, len(true)))
    fig, axes = plt.subplots(len(idxs), 9, figsize=(18, 2.2 * len(idxs)))
    if len(idxs) == 1: axes = np.expand_dims(axes, 0)
    for r, idx in enumerate(idxs):
        t = true[idx]; g = generated[idx]; diff = (g - t).abs()
        vmax = max(float(t.max()), float(g.max()), 1e-6)
        metric = generation_metrics(g.unsqueeze(0), t.unsqueeze(0), include_voxel_auroc=False)
        panels = _middle_slices(t) + _middle_slices(g) + _middle_slices(diff)
        titles = ['true axial','true coronal','true sagittal','gen axial','gen coronal','gen sagittal','diff axial','diff coronal','diff sagittal']
        for c, panel in enumerate(panels):
            axes[r, c].imshow(panel.T, origin='lower', cmap='magma', vmin=0, vmax=vmax if c < 6 else None)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].set_title(titles[c], fontsize=8)
        label = f"test\n{meta.iloc[idx].map_id}\nr={metric['spatial_corr']:.2f} d5={metric['top5_dice']:.2f}"
        axes[r, 0].set_ylabel(label, fontsize=8)
        fig.savefig(out_dir / f"test_{meta.iloc[idx].map_id}.png", dpi=160, bbox_inches='tight')
    fig.tight_layout(); fig.savefig(out_png, dpi=180, bbox_inches='tight')
    return str(out_png), str(out_dir)

test_primary = primary_text_table(rows, text_pairs, 'test')
train_primary = primary_text_table(rows, text_pairs, 'train')
test_target = volumes[test_primary.tensor_index.to_numpy()].float()
train_volumes = volumes[train_primary.tensor_index.to_numpy()].float()

# generated = generate_from_text(test_primary.text.tolist())  # SPECTER -> trained projection -> decoder
# gen_metrics = evaluate_generation_tensors(generated, test_target)
# gen_plot = plot_random_generation_examples(test_target, generated, test_primary, OUT/'random_generation_examples_test.png', OUT/'random_generation_examples_test')

# Baselines to compare against generated maps. These run even before a text-to-brain model is trained.
mean_pred = global_mean_map(train_volumes).unsqueeze(0).expand_as(test_target)
random_pred = random_training_maps(train_volumes, n=len(test_target), seed=SEED)
baselines = {'global_mean_map': mean_pred, 'random_training_map': random_pred}
baseline_metrics = {name: evaluate_generation_tensors(pred, test_target) for name, pred in baselines.items()}
json.dump(baseline_metrics, open(OUT / 'stage4_generation_baselines.json', 'w'), indent=2)
print('Saved generation baselines:', OUT / 'stage4_generation_baselines.json')

if 'generated' in globals():
    gen_metrics = evaluate_generation_tensors(generated, test_target)
    gen_plot = plot_random_generation_examples(test_target, generated, test_primary, OUT/'random_generation_examples_test.png', OUT/'random_generation_examples_test')
else:
    gen_metrics = None
    gen_plot = None
    print('No generated maps found yet. Train/load a text-to-brain projection, create `generated`, then rerun this cell for generation metrics.')

baseline_metrics

## Optional Stage 4b: Train Text-To-Brain Projection Head

Only use train for fitting and val for model selection.

```text
SPECTER/SPECTER2 text -> text-to-brain projection head -> CNN latent -> frozen CNN decoder -> reconstructed ALE volume
```

Loss:

```text
latent_alignment_loss(text_z, brain_z)
+ weighted_reconstruction_loss(decoded, true_ALE)
+ soft_dice_loss(decoded, true_ALE)
+ top_k_overlap_loss(decoded, true_ALE)
+ spatial_correlation_loss(decoded, true_ALE)
```

Then evaluate the selected checkpoint once on test using Stage 4.

In [ ]:
from atlas_free_multipositive.training.train_text_to_brain import train_from_config as train_text_to_brain

t2b_cfg = {
    'train_jsonl': str(SPLIT_JSONL['train']),
    'val_jsonl': str(SPLIT_JSONL['val']),
    'text_embedding_cache': str(CACHE / 'text_embeddings/specter_text_cache.pt'),
    'autoencoder_checkpoint': str(AE_CKPT),
    'target_shape': list(TARGET_SHAPE),
    'output_dir': str(OUT / 'stage4b_text_to_brain'),
    'checkpoint_dir': str(OUT / 'stage4b_text_to_brain/checkpoints'),
    'epochs': 10,
    'batch_size': 8,
    'lr': 1e-4,
    'progress': True,
    'loss': {'lambda_latent': 1.0, 'lambda_recon': 1.0, 'lambda_dice': 0.5, 'lambda_topk': 0.5, 'lambda_corr': 0.25},
    'weighted_recon': {'type': 'mse', 'alpha': 10.0, 'gamma': 1.0},
}

RUN_STAGE4B_TEXT_TO_BRAIN = False
T2B_CKPT = OUT / 'stage4b_text_to_brain/checkpoints/best_generation_top5_dice.pt'
if RUN_STAGE4B_TEXT_TO_BRAIN:
    if not Path(t2b_cfg['text_embedding_cache']).exists():
        raise FileNotFoundError('Stage 4b needs cached SPECTER/SPECTER2 embeddings: ' + t2b_cfg['text_embedding_cache'])
    print('Training optional Stage 4b text-to-brain projection head.')
    t2b_result = train_text_to_brain(t2b_cfg)
    T2B_CKPT = Path(t2b_result['checkpoint_dir']) / 'best_generation_top5_dice.pt'
elif T2B_CKPT.exists():
    print('Using existing Stage 4b checkpoint:', T2B_CKPT)
else:
    print('Optional Stage 4b not run. Set RUN_STAGE4B_TEXT_TO_BRAIN=True after preparing text embeddings.')

T2B_CKPT

## Semantic Evaluation Of Generated Maps

Voxel metrics can understate useful semantic structure for sparse ALE maps. Also evaluate:

```text
text -> generated brain map -> brain encoder -> retrieve/rank text or terms
```

Report generated-map exact PMID retrieval diagnostic, semantic-neighborhood retrieval, MeSH term ranking, and network/term ranking when labels are available.

In [ ]:
from atlas_free_multipositive.evaluation.evaluate_generation_semantic_retrieval import generated_map_retrieval_metrics

# generated_brain_emb = brain_encoder(generated.to(DEVICE)).detach().cpu()
# semantic_generation = generated_map_retrieval_metrics(generated_brain_emb, candidate_text_emb, positive_mask)
# json.dump(semantic_generation, open(OUT/'stage4_generated_semantic_retrieval.json','w'), indent=2)

## Final Summary Table

Save one final table with reconstruction, bidirectional retrieval, text-to-brain generation, baselines, semantic generation diagnostics, and qualitative plot paths.

In [ ]:
summary_rows = []
if 'recon_results' in globals() and recon_results:
    for split, df in recon_results.items():
        means = df.select_dtypes(include='number').mean()
        for metric in ['mse', 'mae', 'spatial_corr', 'top1_dice', 'top5_dice', 'top10_dice', 'voxel_auroc']:
            if metric in means:
                summary_rows.append({'section': 'reconstruction', 'split': split, 'metric': metric, 'value': float(means[metric])})
if 'retrieval_report' in globals():
    for metric, value in retrieval_report.items():
        if isinstance(value, (int, float, np.floating)):
            summary_rows.append({'section': 'retrieval', 'split': 'test', 'metric': metric, 'value': float(value)})
if 'gen_metrics' in globals() and gen_metrics is not None:
    for metric, value in gen_metrics.items():
        summary_rows.append({'section': 'generation', 'split': 'test', 'metric': metric, 'value': float(value)})
if 'baseline_metrics' in globals():
    for name, metrics in baseline_metrics.items():
        for metric, value in metrics.items():
            summary_rows.append({'section': f'generation_baseline:{name}', 'split': 'test', 'metric': metric, 'value': float(value)})
if 'reconstruction_plot_paths' in globals():
    for split, paths in reconstruction_plot_paths.items():
        summary_rows.append({'section': 'qualitative_reconstruction', 'split': split, 'metric': 'plot_png', 'value': paths[0]})
if 'gen_plot' in globals() and gen_plot is not None:
    summary_rows.append({'section': 'qualitative_generation', 'split': 'test', 'metric': 'plot_png', 'value': gen_plot[0]})

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise RuntimeError('No pipeline outputs were collected. Earlier training/evaluation cells did not run.')
summary_path = OUT / 'final_summary_table.csv'
summary.to_csv(summary_path, index=False)
summary